In [6]:
import pandas as pd
from hazm import Normalizer
import re
from rapidfuzz import process, fuzz
import emoji
import numpy as np

# ----------------------------
# تنظیمات اولیه
# ----------------------------
INPUT_CSV  = r'D:\ElMosYar\ElMosYar\extracted_reviews_v3_final.csv'
OUTPUT_CSV = r'extracted_clean_name.csv'

normalizer = Normalizer()

# ----------------------------
# تابع تمیز کردن نام استاد
# ----------------------------
def clean_prof_name_strict(name: str) -> str:
    """
    ورودی: نام خام استاد (professor_name_raw)
    خروجی: نام تمیز و نرمال شده بدون عنوان، بدون ایموجی و نویز
    """
    if pd.isna(name) or not str(name).strip():
        return ""

    name = str(name)

    # حذف ایموجی‌ها
    name = emoji.replace_emoji(name, replace='')

    # حذف بخش‌های بعد از جداکننده‌های خاص
    name = re.split(r'[#|🏫]', name)[0]

    # حذف کلمات نویز
    noise_patterns = [
        r'کانال', r'جزوه', r'امتحان', r'نمره', r'ریاضی', r'فیزیک',
        r'مکانیک', r'عمومی', r'آزمایشگاه', r'زمانی سخت', r'استاتیک'
    ]
    for pat in noise_patterns:
        name = re.sub(pat, ' ', name)

    # نرمال‌سازی حروف فارسی، فاصله و ...
    name = normalizer.normalize(name)

    # حذف عنوان‌ها (بدون استفاده از \b)
    title_pattern = r'(^|\s)(دکتر|دكتر|استاد|پروفسور|پروف|آقای|خانم|مهندس|سید|سیده)(\s|$)'
    for _ in range(3):
        name = re.sub(title_pattern, ' ', name)

    # حذف کاراکترهای غیر حروف/عدد/فاصله
    name = re.sub(r'[^\w\s]', ' ', name)

    # یکی کردن فاصله‌ها
    name = re.sub(r'\s+', ' ', name).strip()

    # اگر خیلی کوتاه شد، بی‌اعتبارش کنیم
    if len(name) < 3:
        return ""

    return name

# ----------------------------
# نگاشت کانُنیکال برای اسامی‌ای که باید حتماً یکی شوند
# ----------------------------
CANONICAL_MAP = {
    # احمد رهبر
    "احمد رهبر کلیشمی": "احمد رهبر کلیشمی",
    "اقای دکتز احمد رهبر": "احمد رهبر کلیشمی",
    "احمد رهبر": "احمد رهبر کلیشمی",

    # امرالهی بیوکی / رضوانه
    "امرالهی بیوکی": "رضوانه امرالهی بیوکی",
    "امر الهی بیوکی": "رضوانه امرالهی بیوکی",
    "امراللهی بیوکی": "رضوانه امرالهی بیوکی",
    "رضوانه امر اللهی بیوکی": "رضوانه امرالهی بیوکی",
    "رضوانه امرالهی بیوکی": "رضوانه امرالهی بیوکی",

    # امیر حسین کردبچه
    "امیر حسین کردبچه": "امیرحسین احمدخان کردبچه",
    "امیرحسین احمدخان کردبچه": "امیرحسین احمدخان کردبچه",

    # امیر سامان نور امین
    "امیر سامان نور امین": "امیرسامان نورامین",
    "امیرسامان نورامین": "امیرسامان نورامین",

    # حسن فرخ زاد
    "حسن فرخ زاد": "حسن فرخزاد",
    "حسن فرخزاد": "حسن فرخزاد",

    # جمال الدین شریعتی
    "جمال الدین شریعتی": "جمال‌الدین شریعتی",
    "جمالالدین شریعتی": "جمال‌الدین شریعتی",

    # حمیدرضا کوه دار
    "حمیدرضا کوه دار": "حمیدرضا کوهدار",
    "حمیدرضا کوهدار": "حمیدرضا کوهدار",

    # رضا نوروز بیگی و نوروزبیگی سعیدزاده عمداً جدا می‌مانند (پس اینجا نمی‌آوریم)

    # محمد حسین / محمدحسین روزبهانی
    "محمد حسین روزبهانی": "محمدحسین روزبهانی",
    "محمدحسین روزبهانی": "محمدحسین روزبهانی",

    # محمد رضا / محمدرضا جعفرفرد
    "محمد رضا جعفر فرد": "محمدرضا جعفرفرد",
    "محمدرضا جعفرفرد": "محمدرضا جعفرفرد",

    # محمد رضا / محمدرضا رسولی
    "محمد رضا رسولی": "محمدرضا رسولی",
    "محمدرضا رسولی": "محمدرضا رسولی",

    # عبد الرسول / عبدارسول
    "عبد الرسول میرقدری": "عبدالرسول میرقدری",
    "عبدارسول میرقدری": "عبدالرسول میرقدری",

    # علی عبد العالی / علی عبدالعالی
    "علی عبد العالی": "علی عبدالعالی",
    "علی عبدالعالی": "علی عبدالعالی",

    # محمد امینی / محمدامین فارسانی
    "محمد امینی فارسانی": "محمدامین فارسانی",
    "محمدامین فارسانی": "محمدامین فارسانی",

    # محمدپور عمران
    "محمدپور عمران": "محمدپورعمران",
    "محمدپورعمران": "محمدپورعمران",

    # مرضیه داودآبادی
    "مرضیه داودآبادی فراهانی": "مرضیه داودآبادی فراهانی",
    "مرضیه داودآبادی": "مرضیه داودآبادی فراهانی",

    # مصطفی مرشد سلوک
    "مصطفی مرشد سلوک": "مصطفی مرشدسلوک",
    "مصطفی مرشدسلوک": "مصطفی مرشدسلوک",

    # نرگس هوشمند
    "نرگس هوشمند": "نرگس هوشمند همدانی",
    "نرگس هوشنمد همدانی": "نرگس هوشمند همدانی",
    "هوشمند همدانی": "نرگس هوشمند همدانی",

    # غلامحسین یاری
    "غلام حسین یاری": "غلامحسین یاری",
    "غلامحسین یاری": "غلامحسین یاری",

    # زهرا تند پور (یکسان‌سازی فاصله)
    "زهرا تند پور": "زهرا تندپور",
    "زهرا تندپور": "زهرا تندپور",
}

CANONICAL_TARGETS = set(CANONICAL_MAP.values())

def apply_canonical_overrides(name: str) -> str:
    return CANONICAL_MAP.get(name, name)

# ----------------------------
# خواندن داده
# ----------------------------
print("Loading data...")
df = pd.read_csv(INPUT_CSV)

if 'professor_name_raw' not in df.columns:
    raise ValueError("Column 'professor_name_raw' not found in CSV.")

# مرحله 1: تمیز کردن نام‌ها
print("1. Cleaning professor names ...")
df['clean_key'] = df['professor_name_raw'].apply(clean_prof_name_strict)

df = df[df['clean_key'] != ""].copy()
df.reset_index(drop=True, inplace=True)
print(f"Remaining rows after cleaning: {len(df)}")

# اعمال نگاشت کانُنیکال
df['clean_key'] = df['clean_key'].apply(apply_canonical_overrides)

# ----------------------------
# مرحله 2: خوشه‌بندی سریع (فازی)
# ----------------------------
unique_names = df['clean_key'].unique().tolist()
unique_names.sort(key=len, reverse=True)

print(f"2. Clustering {len(unique_names)} unique cleaned names (Fast Mode)...")

name_to_id = {}
processed = set()
current_id = 0

for name in unique_names:
    if name in processed:
        continue

    name_to_id[name] = current_id
    processed.add(name)

    matches = process.extract(
        name,
        unique_names,
        scorer=fuzz.token_set_ratio,
        limit=20,
        score_cutoff=78
    )

    for match_name, score, _ in matches:
        if match_name not in processed:
            name_to_id[match_name] = current_id
            processed.add(match_name)

    current_id += 1

df['professor_id'] = df['clean_key'].map(name_to_id)
print(f"Initial unique professors (clusters): {df['professor_id'].nunique()}")

# ----------------------------
# مرحله 3: ادغام خوشه‌های subset
# ----------------------------
print("3. Merging subset name clusters...")

id_to_rep = {}
for pid, group in df.groupby('professor_id'):
    longest = max(group['clean_key'].unique(), key=len)
    id_to_rep[pid] = longest

remapping = {}
sorted_ids = sorted(id_to_rep.keys(), key=lambda k: len(id_to_rep[k]), reverse=True)

for pid_long in sorted_ids:
    long_name = id_to_rep[pid_long].replace(" ", "")

    # اگر نماینده‌ی خوشه، یکی از targetهای کانُنیکال است، دست‌کاری نکن
    if id_to_rep[pid_long] in CANONICAL_TARGETS:
        continue

    for pid_short in sorted_ids:
        if pid_long == pid_short:
            continue

        short_name = id_to_rep[pid_short].replace(" ", "")

        if len(short_name) > 4 and short_name in long_name:
            # اگر کوتاهه هم جزو targetهای کانُنیکال باشد، ادغام نکن
            if id_to_rep[pid_short] in CANONICAL_TARGETS:
                continue
            remapping[pid_short] = pid_long

df['professor_id'] = df['professor_id'].replace(remapping)
print(f"Unique professors after subset merge: {df['professor_id'].nunique()}")

# ----------------------------
# مرحله 4: انتخاب نام استاندارد از clean_key
# ----------------------------
def get_best_display_name_clean(series):
    candidates = [c for c in series.unique().tolist() if isinstance(c, str)]
    if not candidates:
        return ""
    return max(candidates, key=len)

print("4. Selecting standard display names from cleaned names...")
standard_names_clean = (
    df.groupby('professor_id')['clean_key']
      .apply(get_best_display_name_clean)
      .to_dict()
)

df['professor_name_standard'] = df['professor_id'].map(standard_names_clean)

# ----------------------------
# مرحله 5: ذخیره خروجی
# ----------------------------
print("5. Saving outputs...")

mapping_df = (
    df[['professor_name_raw', 'professor_name_standard', 'professor_id']]
    .drop_duplicates()
    .sort_values('professor_id')
)

mapping_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print("Done!")
print(f"Final unique professors: {df['professor_id'].nunique()}")

# تست موردی
check_mask = df['professor_name_raw'].astype(str).str.contains('فارسانی', na=False)
print("\nCheck Results for 'فارسانی':")
print(
    df[check_mask][['professor_name_raw', 'professor_name_standard', 'professor_id']]
    .drop_duplicates()
    .head(20)
)


Loading data...
1. Cleaning professor names ...
Remaining rows after cleaning: 4548
2. Clustering 1125 unique cleaned names (Fast Mode)...
Initial unique professors (clusters): 638
3. Merging subset name clusters...
Unique professors after subset merge: 623
4. Selecting standard display names from cleaned names...
5. Saving outputs...
Done!
Final unique professors: 623

Check Results for 'فارسانی':
            professor_name_raw professor_name_standard  professor_id
33          محمد امینی فارسانی        محمدامین فارسانی           152
272   استاد محمد امینی فارسانی        محمدامین فارسانی           152
482           محمدامین فارسانی        محمدامین فارسانی           152
744                    فارسانی      محمد فارسانی امینی            92
808              استاد فارسانی      محمد فارسانی امینی            92
2018        محمد فارسانی امینی      محمد فارسانی امینی            92
2116         محمد امین فارسانی      محمد فارسانی امینی            92


In [4]:
import pandas as pd
from hazm import Normalizer
import re
from rapidfuzz import process, fuzz
import emoji
import numpy as np

# ----------------------------
# تنظیمات اولیه
# ----------------------------
INPUT_CSV  = r'D:\ElMosYar\ElMosYar\extracted_reviews_v3_final.csv'
OUTPUT_FULL_CSV = r'full_cleaned_reviews.csv'  # New: Save full DF here

normalizer = Normalizer()

# ----------------------------
# تابع تمیز کردن نام استاد
# ----------------------------
def clean_prof_name_strict(name: str) -> str:
    """
    ورودی: نام خام استاد (professor_name_raw)
    خروجی: نام تمیز و نرمال شده بدون عنوان، بدون ایموجی و نویز
    """
    if pd.isna(name) or not str(name).strip():
        return ""

    name = str(name)

    # حذف ایموجی‌ها
    name = emoji.replace_emoji(name, replace='')

    # حذف بخش‌های بعد از جداکننده‌های خاص
    name = re.split(r'[#|🏫]', name)[0]

    # حذف کلمات نویز
    noise_patterns = [
        r'کانال', r'جزوه', r'امتحان', r'نمره', r'ریاضی', r'فیزیک',
        r'مکانیک', r'عمومی', r'آزمایشگاه', r'زمانی سخت', r'استاتیک'
    ]
    for pat in noise_patterns:
        name = re.sub(pat, ' ', name)

    # نرمال‌سازی حروف فارسی، فاصله و ...
    name = normalizer.normalize(name)

    # حذف عنوان‌ها (بدون استفاده از \b)
    title_pattern = r'(^|\s)(دکتر|دكتر|استاد|پروفسور|پروف|آقای|خانم|مهندس|سید|سیده)(\s|$)'
    for _ in range(3):
        name = re.sub(title_pattern, ' ', name)

    # حذف کاراکترهای غیر حروف/عدد/فاصله
    name = re.sub(r'[^\w\s]', ' ', name)

    # یکی کردن فاصله‌ها
    name = re.sub(r'\s+', ' ', name).strip()

    # اگر خیلی کوتاه شد، بی‌اعتبارش کنیم
    if len(name) < 3:
        return ""

    return name

# ----------------------------
# نگاشت کانُنیکال برای اسامی‌ای که باید حتماً یکی شوند
# ----------------------------
CANONICAL_MAP = {
    # احمد رهبر
    "احمد رهبر کلیشمی": "احمد رهبر کلیشمی",
    "اقای دکتز احمد رهبر": "احمد رهبر کلیشمی",
    "احمد رهبر": "احمد رهبر کلیشمی",

    # امرالهی بیوکی / رضوانه
    "امرالهی بیوکی": "رضوانه امرالهی بیوکی",
    "امر الهی بیوکی": "رضوانه امرالهی بیوکی",
    "امراللهی بیوکی": "رضوانه امرالهی بیوکی",
    "رضوانه امر اللهی بیوکی": "رضوانه امرالهی بیوکی",
    "رضوانه امرالهی بیوکی": "رضوانه امرالهی بیوکی",
    "امرالهی":"رضوانه امرالهی بیوکی",

    # امیر حسین کردبچه
    "امیر حسین کردبچه": "امیرحسین احمدخان کردبچه",
    "امیرحسین احمدخان کردبچه": "امیرحسین احمدخان کردبچه",

    # امیر سامان نور امین
    "امیر سامان نور امین": "امیرسامان نورامین",
    "امیرسامان نورامین": "امیرسامان نورامین",

    # حسن فرخ زاد
    "حسن فرخ زاد": "حسن فرخزاد",
    "حسن فرخزاد": "حسن فرخزاد",

    # جمال الدین شریعتی
    "جمال الدین شریعتی": "جمال‌الدین شریعتی",
    "جمالالدین شریعتی": "جمال‌الدین شریعتی",

    # حمیدرضا کوه دار
    "حمیدرضا کوه دار": "حمیدرضا کوهدار",
    "حمیدرضا کوهدار": "حمیدرضا کوهدار",

    # رضا نوروز بیگی و نوروزبیگی سعیدزاده عمداً جدا می‌مانند (پس اینجا نمی‌آوریم)

    # محمد حسین / محمدحسین روزبهانی
    "محمد حسین روزبهانی": "محمدحسین روزبهانی",
    "محمدحسین روزبهانی": "محمدحسین روزبهانی",

    # محمد رضا / محمدرضا جعفرفرد
    "محمد رضا جعفر فرد": "محمدرضا جعفرفرد",
    "محمدرضا جعفرفرد": "محمدرضا جعفرفرد",

    # محمد رضا / محمدرضا رسولی
    "محمد رضا رسولی": "محمدرضا رسولی",
    "محمدرضا رسولی": "محمدرضا رسولی",

    # عبد الرسول / عبدارسول
    "عبد الرسول میرقدری": "عبدالرسول میرقدری",
    "عبدارسول میرقدری": "عبدالرسول میرقدری",

    # علی عبد العالی / علی عبدالعالی
    "علی عبد العالی": "علی عبدالعالی",
    "علی عبدالعالی": "علی عبدالعالی",

    # محمد امینی / محمدامین فارسانی
    "محمد امینی فارسانی": "محمدامین فارسانی",
    "محمدامین فارسانی": "محمدامین فارسانی",

    # محمدپور عمران
    "محمدپور عمران": "محمدپورعمران",
    "محمدپورعمران": "محمدپورعمران",

    # مرضیه داودآبادی
    "مرضیه داودآبادی فراهانی": "مرضیه داودآبادی فراهانی",
    "مرضیه داودآبادی": "مرضیه داودآبادی فراهانی",

    # مصطفی مرشد سلوک
    "مصطفی مرشد سلوک": "مصطفی مرشدسلوک",
    "مصطفی مرشدسلوک": "مصطفی مرشدسلوک",

    # نرگس هوشمند
    "نرگس هوشمند": "نرگس هوشمند همدانی",
    "نرگس هوشنمد همدانی": "نرگس هوشمند همدانی",
    "هوشمند همدانی": "نرگس هوشمند همدانی",

    # غلامحسین یاری
    "غلام حسین یاری": "غلامحسین یاری",
    "غلامحسین یاری": "غلامحسین یاری",

    # زهرا تند پور (یکسان‌سازی فاصله)
    "زهرا تند پور": "زهرا تندپور",
    "زهرا تندپور": "زهرا تندپور",
}

CANONICAL_TARGETS = set(CANONICAL_MAP.values())

def apply_canonical_overrides(name: str) -> str:
    return CANONICAL_MAP.get(name, name)

# ----------------------------
# خواندن داده
# ----------------------------
print("Loading data...")
df = pd.read_csv(INPUT_CSV)

if 'professor_name_raw' not in df.columns:
    raise ValueError("Column 'professor_name_raw' not found in CSV.")

# مرحله 1: تمیز کردن نام‌ها
print("1. Cleaning professor names ...")
df['clean_key'] = df['professor_name_raw'].apply(clean_prof_name_strict)

df = df[df['clean_key'] != ""].copy()
df.reset_index(drop=True, inplace=True)
print(f"Remaining rows after cleaning: {len(df)}")

# اعمال نگاشت کانُنیکال
df['clean_key'] = df['clean_key'].apply(apply_canonical_overrides)

# ----------------------------
# مرحله 2: خوشه‌بندی سریع (فازی)
# ----------------------------
unique_names = df['clean_key'].unique().tolist()
unique_names.sort(key=len, reverse=True)

print(f"2. Clustering {len(unique_names)} unique cleaned names (Fast Mode)...")

name_to_id = {}
processed = set()
current_id = 0

for name in unique_names:
    if name in processed:
        continue

    name_to_id[name] = current_id
    processed.add(name)

    matches = process.extract(
        name,
        unique_names,
        scorer=fuzz.token_set_ratio,
        limit=20,
        score_cutoff=78
    )

    for match_name, score, _ in matches:
        if match_name not in processed:
            name_to_id[match_name] = current_id
            processed.add(match_name)

    current_id += 1

df['professor_id'] = df['clean_key'].map(name_to_id)
print(f"Initial unique professors (clusters): {df['professor_id'].nunique()}")

# ----------------------------
# مرحله 3: ادغام خوشه‌های subset
# ----------------------------
print("3. Merging subset name clusters...")

id_to_rep = {}
for pid, group in df.groupby('professor_id'):
    longest = max(group['clean_key'].unique(), key=len)
    id_to_rep[pid] = longest

remapping = {}
sorted_ids = sorted(id_to_rep.keys(), key=lambda k: len(id_to_rep[k]), reverse=True)

for pid_long in sorted_ids:
    long_name = id_to_rep[pid_long].replace(" ", "")

    # اگر نماینده‌ی خوشه، یکی از targetهای کانُنیکال است، دست‌کاری نکن
    if id_to_rep[pid_long] in CANONICAL_TARGETS:
        continue

    for pid_short in sorted_ids:
        if pid_long == pid_short:
            continue

        short_name = id_to_rep[pid_short].replace(" ", "")

        if len(short_name) > 4 and short_name in long_name:
            # اگر کوتاهه هم جزو targetهای کانُنیکال باشد، ادغام نکن
            if id_to_rep[pid_short] in CANONICAL_TARGETS:
                continue
            remapping[pid_short] = pid_long

df['professor_id'] = df['professor_id'].replace(remapping)
print(f"Unique professors after subset merge: {df['professor_id'].nunique()}")

# ----------------------------
# مرحله 4: انتخاب نام استاندارد از clean_key
# ----------------------------
def get_best_display_name_clean(series):
    candidates = [c for c in series.unique().tolist() if isinstance(c, str)]
    if not candidates:
        return ""
    return max(candidates, key=len)

print("4. Selecting standard display names from cleaned names...")
standard_names_clean = (
    df.groupby('professor_id')['clean_key']
      .apply(get_best_display_name_clean)
      .to_dict()
)

df['professor_name_standard'] = df['professor_id'].map(standard_names_clean)

# ----------------------------
# مرحله 5: ذخیره خروجی (حالا کل DF را ذخیره می‌کنیم)
# ----------------------------
print("5. Saving full cleaned DF...")
df.to_csv(OUTPUT_FULL_CSV, index=False, encoding='utf-8-sig')

print("Done!")
print(f"Final unique professors: {df['professor_id'].nunique()}")

# تست موردی
check_mask = df['professor_name_raw'].astype(str).str.contains('فارسانی', na=False)
print("\nCheck Results for 'فارسانی':")
print(
    df[check_mask][['professor_name_raw', 'professor_name_standard', 'professor_id']]
    .drop_duplicates()
    .head(20)
)

Loading data...
1. Cleaning professor names ...
Remaining rows after cleaning: 4548
2. Clustering 1124 unique cleaned names (Fast Mode)...
Initial unique professors (clusters): 637
3. Merging subset name clusters...
Unique professors after subset merge: 622
4. Selecting standard display names from cleaned names...
5. Saving full cleaned DF...
Done!
Final unique professors: 622

Check Results for 'فارسانی':
            professor_name_raw professor_name_standard  professor_id
33          محمد امینی فارسانی        محمدامین فارسانی           152
272   استاد محمد امینی فارسانی        محمدامین فارسانی           152
482           محمدامین فارسانی        محمدامین فارسانی           152
744                    فارسانی      محمد فارسانی امینی            92
808              استاد فارسانی      محمد فارسانی امینی            92
2018        محمد فارسانی امینی      محمد فارسانی امینی            92
2116         محمد امین فارسانی      محمد فارسانی امینی            92


In [7]:
len(df)

4548

In [5]:
# این خط همه نام‌ها رو چاپ می‌کنه بدون محدودیت طول
print(len(df['professor_name_standard']),len(df['professor_name_standard'].unique()))
# print("\n".join(df['professor_name_standard'].unique().tolist()))

# یا اگر می‌خوای مرتب‌شده و شماره‌دار ببینی (خیلی خواناتر)
names = sorted(df['professor_name_standard'].unique())
for i, name in enumerate(names, 1):
    print(f"{i:3d}. {name}")

4548 622
  1. آتنا بروغنی
  2. آذری
  3. آرش سراحیان
  4. آرش کیومرثی
  5. آرمیتا زاهدی
  6. آرنگ پژوهشگر
  7. آروین قاسمی
  8. آزاده تجردی
  9. آسیه سادات کاظمی
 10. آسیه نمازی
 11. آویده کامرانی
 12. آیت قره قانی
 13. ابراهیم اردشیر لاریجانی
 14. ابراهیم تیموری
 15. ابن الرسول
 16. ابوالفضل اسکندری
 17. ابوالفضل حسینی زاده
 18. ابوالفضل دیانت
 19. ابوذر غفاری
 20. احد قائمی
 21. احسان اکبری
 22. احسان دارستانی فراهانی
 23. احسان دهقانی برسیانی
 24. احمد آیت
 25. احمد رضا رحیمی
 26. احمد رضا شاه علی
 27. احمد رضا فروغ
 28. احمد رهبر کلیشمی
 29. احمد ماکویی
 30. احمد چلداوی
 31. ادریس فیض آبادی
 32. ادیب ابریشمی فر
 33. اساتید سلحشور و جعفرفرد
 34. اساتید سیدین و ابوطالبی
 35. اسدالله آقاجانی
 36. اسماء سلیمانی
 37. اسماعیل اسفندیاری
 38. اسماعیل خان میرزا
 39. اسمعیلی
 40. اصغر محمدمرادی
 41. افسانه ملاحسینی
 42. افشین شریعت مهیمنی
 43. افشین نمیرانیان
 44. المیرا جعفری پور
 45. الهام اله مرادی
 46. الهام رجبی
 47. الهام فتاحی
 48. الهه عرب زاده
 49. امید نیکان
 50. امیر غفاری
 51. ام

In [6]:
df.tail()

,professor_name_raw,department,course_name,sources,attendance,exam_sufficient,grading,rating_1,rating_2,rating_3,rating_4,rating_5,rating_6,contact,term,comments,clean_key,professor_id,professor_name_standard
4543,سید حمیدرضا نوشکاران,ریاضی,معادلات دیفرانسیل,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,بله,منصفانه و هرچی خودت بگیری,8.0,8.0,8.0,9.0,8.0,10.0,ایمیل,بهمن 1403,ترم اولی بود که تدریس می‌کردن ولی تدریس بدی ند...,حمیدرضا نوشکاران,51,سیدحمیدرضا نوش کاران
4544,عین الله خنجری,مهندسی_کامپیوتر,پایگاه داده پیشرفته,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,نمره خوبی نمیشه ازشون گرفت,8.0,6.0,4.0,6.0,1.0,3.0,چیزی ندارم,بهمن ۱۴۰۳,تو نمره دادن پروژه‌ها بسیار رندوم و بی‌منطق عم...,عین الله خنجری,300,عین الله خنجری
4545,رضا انتظاری ملکی,مهندسی_کامپیوتر,مباحث ویژه نرم افزار ۱,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,چیزی ندارم,بهمن ۱۴۰۳,استاد بسیار خوش اخلاق و محترمی هستند. تعداد تم...,رضا انتظاری ملکی,158,رضا انتظاری ملکی
4546,دکتر بخشایش,مهندسی_مواد,مواد پیشرفته,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,بله شماره تلفن همراه,بهمن 1401,استاد مطلعی هستند و اطلاعات به روزی دارند.,بخشایش,186,امیرمحمود بخشایش
4547,فاطمه مهدی زاده سراج,معماری_و_شهرسازی,مبانی طراحی ۲,منبعی معرفی نمی کنند,یادم نمیاد,بله,منصفانه و هرچی خودت بگیری,10.0,10.0,10.0,10.0,10.0,10.0,چیزی ندارم,بهمن ۱۴۰۳,با دکتر طباطبایی و دو سه تا تی ای میان سر کلاس...,فاطمه مهدی زاده سراج,64,فاطمه مهدی زاده سراج
